
Runs after `KNN_model.ipynb` so that `session_log.json` and `knn_model.joblib` are created. 

session_log.json ──► for each answer below 80%:
                       RETRIEVE  1) the model answer (exact lookup in the dataset)
                                 2) the 3 most similar dataset questions in the same career (TF-IDF cosine)
                              ▼
                       GENERATE  feedback report: strengths, weak topics, missing key terms, study plan
                              ▼
                       feedback_report.json (for the app) + feedback_report.txt (readable)

It is offline  by default (rule-based, grounded only in the retrieved dataset text) and writes the overall coaching summary from the same context.

 1. Imports and settings

In [1]:
import json, os, re
import numpy as np, pandas as pd, joblib
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

SESSION_FILE = "session_log.json"      # written by KNN_model.ipynb
MODEL_FILE   = "knn_model.joblib"      # written by KNN_model.ipynb
REPORT_JSON  = "feedback_report.json"
REPORT_TXT   = "feedback_report.txt"
TOP_RELATED  = 3                       # related study questions retrieved per weak answer
MIN_SIMILARITY = 0.05                  # ignore dataset questions that share almost nothing with the weak one
STRONG, PASS = 80, 50                  # score thresholds (same as the KNN notebook)
LEVELS       = ["Easy", "Medium", "Hard"]
LLM_MODEL    = "claude-sonnet-5-5"     # only used if ANTHROPIC_API_KEY is set


 2. Load the interview log and the question dataset

In [2]:
with open(SESSION_FILE, encoding="utf-8") as f:
    session = json.load(f)

knn_df = joblib.load(MODEL_FILE)["knn_df"].reset_index(drop=True)

print("Career:", session["career"], "| answers:", session["questions_answered"],
      "| average score:", session["average_score"])
print("Knowledge base:", len(knn_df), "questions")


Career: Software Engineering | answers: 1 | average score: 28.0
Knowledge base: 204 questions


 3. Build the retrieval index (the knowledge base is the question dataset)

In [3]:
kb_text = knn_df["Question"] + " " + knn_df["Category"] + " " + knn_df["Answer"]

kb_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
kb_matrix = kb_vectorizer.fit_transform(kb_text)
kb_matrix.shape


(204, 4244)

 4. Retrieval functions

In [4]:
def get_expected_answer(question):
    """Retrieval 1: exact lookup of the model answer."""
    row = knn_df[knn_df["Question"] == question]
    return row.iloc[0]["Answer"] if not row.empty else None


def retrieve_related(question, career, k=TOP_RELATED):
    """Retrieval 2: dataset questions related to this one (same career, excluding itself).
    Same-category questions come first; others must be similar enough (>= MIN_SIMILARITY)."""
    row = knn_df[knn_df["Question"] == question]
    category = row.iloc[0]["Category"] if not row.empty else None
    expected = row.iloc[0]["Answer"] if not row.empty else ""

    query = kb_vectorizer.transform([question + " " + expected])
    sims = cosine_similarity(query, kb_matrix)[0]

    cand = knn_df.assign(similarity=sims)
    cand = cand[(cand["Question"] != question) & (cand["Career"] == career)]
    cand = cand.assign(same_category=(cand["Category"] == category))
    cand = cand[cand["same_category"] | (cand["similarity"] >= MIN_SIMILARITY)]
    cand = cand.sort_values(["same_category", "similarity"], ascending=False).head(k)

    return [
        {"question": r["Question"], "category": r["Category"], "difficulty": r["Difficulty"],
         "answer": r["Answer"], "similarity": round(float(r["similarity"]), 3)}
        for _, r in cand.iterrows()
    ]


5. Helpers: missing key terms and verdicts

In [5]:
# these words are not in the sklearn stop word list, but they are common in model answers 
# and not useful for scoring

EXTRA_STOP = {"allows", "using", "used", "also", "like", "often", "such", "based", "called",
              "between", "within", "typically", "usually", "different", "various"}

def key_terms(text):
    """Important words of a text, in order, without duplicates."""
    seen, terms = set(), []
    for w in re.findall(r"[a-zA-Z][a-zA-Z\-]{3,}", str(text).lower()):
        if w in ENGLISH_STOP_WORDS or w in EXTRA_STOP or w in seen:
            continue
        seen.add(w)
        terms.append(w)
    return terms


def missing_terms(expected, user_answer, limit=8):
    """Key terms of the model answer that the user did not mention."""
    user_stems = {w[:5] for w in key_terms(user_answer)}
    return [w for w in key_terms(expected) if w[:5] not in user_stems][:limit]


def verdict(score):
    if score >= STRONG:
        return "Strong"
    if score >= PASS:
        return "Partly correct"
    return "Needs work"


def performance_level(avg):
    if avg >= 70:
        return "Strong"
    if avg >= PASS:
        return "Developing"
    return "Needs work"


6. Feedback for one answer (retrieval + rules)

In [6]:
# score feedback for a single question, including related questions to study next
def question_feedback(entry, career):
    expected = entry.get("expected_answer") or get_expected_answer(entry["question"]) or ""
    user_answer = entry.get("user_answer", "")
    score = entry["score"]
    v = verdict(score)
    weak = score < STRONG

    if not str(user_answer).strip():
        text = "No answer was given. Read the model answer and try again."
    elif v == "Strong":
        text = "Strong answer. It covers the main points of the model answer."
    elif v == "Partly correct":
        text = "Partly correct. Add the missing key terms and compare your answer with the model answer."
    else:
        text = "This answer needs work. Study the model answer and the related questions below."

    return {
        "question": entry["question"],
        "category": entry.get("category"),
        "difficulty": entry.get("difficulty"),
        "score": score,
        "verdict": v,
        "your_answer": user_answer,
        "model_answer": expected,
        "missing_terms": missing_terms(expected, user_answer) if weak else [],
        "feedback": text,
        "study_next": retrieve_related(entry["question"], career) if weak else [],
    }


7. Generators for the overall summary   :  
Offline (rule-based) by default. The LLM option uses only the retrieved context.

In [7]:
# summarize the interview results in a few sentences, with optional LLM feedback if available
def rule_based_narrative(r):
    if r["questions_answered"] == 0:
        return "No questions were answered, so there is no feedback yet."
    parts = [f"You answered {r['questions_answered']} {r['career']} questions with an average score of "
             f"{r['average_score']}% ({r['performance_level']})."]
    if r["hardest_level_passed"]:
        parts.append(f"The hardest level you answered well was {r['hardest_level_passed']}.")
    if r["strong_topics"]:
        parts.append("Strong topics: " + ", ".join(r["strong_topics"]) + ".")
    if r["weak_topics"]:
        parts.append("Revise these first: " + ", ".join(w["category"] for w in r["weak_topics"][:3]) + ".")
    else:
        parts.append("No topic fell below the pass mark. Try another interview to reach harder questions.")
    return " ".join(parts)


def llm_narrative(r):
    """Optional: an LLM writes the coaching summary from the retrieved context only."""
    key = os.getenv("ANTHROPIC_API_KEY")
    if not key:
        return None
    context = {
        "career": r["career"], "average_score": r["average_score"], "level": r["performance_level"],
        "strong_topics": r["strong_topics"], "weak_topics": r["weak_topics"],
        "answers": [{"question": q["question"], "score": q["score"], "missing_terms": q["missing_terms"],
                     "model_answer": q["model_answer"],
                     "related_questions": [s["question"] for s in q["study_next"]]}
                    for q in r["questions"]],
    }
    prompt = ("You are an interview coach for IT students. Write feedback for the candidate using ONLY the "
              "CONTEXT below (taken from the interview log and the question dataset). Do not invent facts or "
              "questions. Format: a 3-4 sentence overall assessment, then 'Top priorities:' with 3 short "
              "bullet points. Address the candidate as 'you'.\n\nCONTEXT:\n" + json.dumps(context, indent=1))
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=key)
        msg = client.messages.create(model=LLM_MODEL, max_tokens=700,
                                     messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text.strip()
    except Exception as e:
        print("LLM unavailable, using the offline feedback instead:", type(e).__name__)
        return None


8. Build the full report

In [8]:
# build the final report from the session log and the retrieved context
def build_report(session):
    log = session.get("log", [])
    career = session.get("career")
    by_cat = session.get("average_by_category", {})
    questions = [question_feedback(e, career) for e in log]

    weak_topics = sorted(
        [{"category": c, "average_score": s} for c, s in by_cat.items() if s < PASS],
        key=lambda w: w["average_score"])
    strong_topics = [c for c, s in by_cat.items() if s >= STRONG]

    passed = [q["difficulty"] for q in questions if q["score"] >= PASS and q["difficulty"] in LEVELS]
    hardest = max(passed, key=LEVELS.index) if passed else None

    # study plan: weakest topics first, using the retrieved related questions
    plan = []
    for w in weak_topics:
        related = []
        for q in questions:
            if q["category"] == w["category"]:
                related += [s["question"] for s in q["study_next"] if s["question"] not in related]
        line = f"Revise {w['category']} (average {w['average_score']:g}%)."
        if related:
            line += " Practise: " + "; ".join(related[:3])
        plan.append(line)
    if not plan and questions:
        plan.append("Keep practising and aim for Hard questions in your next interview.")

    report = {
        "career": career,
        "questions_answered": len(log),
        "average_score": session.get("average_score", 0),
        "performance_level": performance_level(session.get("average_score", 0)),
        "hardest_level_passed": hardest,
        "strong_topics": strong_topics,
        "weak_topics": weak_topics,
        "questions": questions,
        "study_plan": plan,
    }
    llm_text = llm_narrative(report)
    report["summary"] = llm_text or rule_based_narrative(report)
    report["generated_by"] = "LLM + retrieved context" if llm_text else "offline rules + retrieved context"
    return report


 9. Show and save the report

In [9]:
# format the report as a text file
def format_report(r):
    L = ["=" * 64, f"INTERVIEW FEEDBACK: {r['career']}", "=" * 64,
         f"Average score: {r['average_score']}%  |  Level: {r['performance_level']}  |  Answers: {r['questions_answered']}",
         "", r["summary"], ""]
    if r["study_plan"]:
        L += ["STUDY PLAN"] + [f"  {i}. {s}" for i, s in enumerate(r["study_plan"], 1)] + [""]
    L.append("QUESTION BY QUESTION")
    for i, q in enumerate(r["questions"], 1):
        L += ["-" * 64, f"{i}. [{q['difficulty']}] {q['question']}", f"   Score: {q['score']:g}/100 ({q['verdict']})",
              f"   Your answer: {q['your_answer'] or '(none)'}", f"   Feedback: {q['feedback']}"]
        if q["missing_terms"]:
            L.append("   Missing key terms: " + ", ".join(q["missing_terms"]))
        if q["verdict"] != "Strong":
            L.append(f"   Model answer: {q['model_answer']}")
            if not q["study_next"]:
                L.append("   Study next: no closely related question in the dataset. Re-read the model answer.")
        for s in q["study_next"]:
            L.append(f"   Study next: {s['question']} ({s['category']}, {s['difficulty']}, similarity {s['similarity']})")
    L += ["", f"Generated by: {r['generated_by']}"]
    return "\n".join(L)


def generate_feedback(session_file=SESSION_FILE):
    with open(session_file, encoding="utf-8") as f:
        sess = json.load(f)
    report = build_report(sess)
    text = format_report(report)
    with open(REPORT_JSON, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    with open(REPORT_TXT, "w", encoding="utf-8") as f:
        f.write(text)
    print(text)
    print(f"\nSaved {REPORT_JSON} and {REPORT_TXT}")
    return report


 10. Generate the feedback report

In [10]:
report = generate_feedback()


INTERVIEW FEEDBACK: Software Engineering
Average score: 28.0%  |  Level: Needs work  |  Answers: 1

You answered 1 Software Engineering questions with an average score of 28.0% (Needs work). Revise these first: Software Development Tools.

STUDY PLAN
  1. Revise Software Development Tools (average 28%). Practise: What is version control?; What is Git, and how does it work?; Explain the difference between Git and SVN (Subversion).

QUESTION BY QUESTION
----------------------------------------------------------------
1. [Easy] What is Git?
   Score: 28/100 (Needs work)
   Your answer: Version Control
   Feedback: This answer needs work. Study the model answer and the related questions below.
   Missing key terms: distributed, track, changes, files, coordinate, software, development
   Model answer: Git is a distributed version control system used to track changes in files and coordinate software development.
   Study next: What is version control? (Software Development Tools, Easy, simil

11. Tests

In [11]:
# Simple tests to check the report structure and content. 
# They catch common mistakes.
def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    return ok

res = []
res.append(check("Report has all sections", all(k in report for k in
      ["summary", "study_plan", "questions", "strong_topics", "weak_topics", "average_score"])))
res.append(check("One feedback item per answered question", len(report["questions"]) == session["questions_answered"]))
res.append(check("Study questions are relevant (same category or similar enough)",
      all(s["category"] == q["category"] or s["similarity"] >= MIN_SIMILARITY
          for q in report["questions"] for s in q["study_next"])))
res.append(check("Study questions stay in the chosen career",
      all(s["category"] in set(knn_df[knn_df["Career"] == session["career"]]["Category"])
          for q in report["questions"] for s in q["study_next"])))
res.append(check("Strong answers need no extra study questions",
      all(q["study_next"] == [] for q in report["questions"] if q["score"] >= STRONG)))
res.append(check("Related questions never repeat the question itself",
      all(s["question"] != q["question"] for q in report["questions"] for s in q["study_next"])))
res.append(check("Related questions come from the dataset",
      all(s["question"] in set(knn_df["Question"]) for q in report["questions"] for s in q["study_next"])))
res.append(check("Model answer matches the dataset",
      all(q["model_answer"] == get_expected_answer(q["question"]) for q in report["questions"])))
res.append(check("A blank answer is handled",
      question_feedback({"question": knn_df.iloc[0]["Question"], "user_answer": "", "score": 0,
                         "category": "x", "difficulty": "Easy"}, session["career"])["feedback"].startswith("No answer")))
res.append(check("Empty interview is handled",
      "No questions were answered" in build_report({"career": "Web Development", "log": [], "average_score": 0})["summary"]))
res.append(check("Report files saved", os.path.exists(REPORT_JSON) and os.path.exists(REPORT_TXT)))
print(f"\n{sum(res)}/{len(res)} tests passed")


PASS  Report has all sections
PASS  One feedback item per answered question
PASS  Study questions are relevant (same category or similar enough)
PASS  Study questions stay in the chosen career
PASS  Strong answers need no extra study questions
PASS  Related questions never repeat the question itself
PASS  Related questions come from the dataset
PASS  Model answer matches the dataset
PASS  A blank answer is handled
PASS  Empty interview is handled
PASS  Report files saved

11/11 tests passed
